In [1]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import os
import requests
import unicodedata
import re

from pathlib import Path

In [2]:
# ============================================================
# 2. LOAD TEAMS
# ============================================================

team_master = pd.read_csv("team_master.csv")

teams = sorted(team_master["team"].dropna().unique())

print(len(teams))
teams[:10]

71


['Albania',
 'Algeria',
 'Angola',
 'Argentina',
 'Australia',
 'Austria',
 'Belgium',
 'Bolivia',
 'Brazil',
 'Burkina Faso']

In [3]:
# ============================================================
# 3. CREATE ASSET FOLDERS
# ============================================================

assets_path = Path("streamlit_app/assets")
flags_path = assets_path / "flags"

flags_path.mkdir(parents=True, exist_ok=True)

print("Asset folders ready!")

Asset folders ready!


In [4]:
# ============================================================
# 4. CLEAN FILE NAMES
# ============================================================

def clean_name(name):

    name = unicodedata.normalize("NFKD", name)

    name = "".join(
        [
            c for c in name
            if not unicodedata.combining(c)
        ]
    )

    name = name.lower()
    name = name.replace("&", "and")

    name = re.sub(
        r"[^a-z0-9]+",
        "_",
        name
    )

    name = name.strip("_")

    return name

In [5]:
# ============================================================
# 5. TEAM COLORS AND COUNTRY CODES
# ============================================================

team_info = {
    "Argentina": {"primary_color": "#75AADB", "secondary_color": "#FFFFFF", "code": "AR"},
    "France": {"primary_color": "#1D3557", "secondary_color": "#FFFFFF", "code": "FR"},
    "Brazil": {"primary_color": "#009C3B", "secondary_color": "#FFDF00", "code": "BR"},
    "Spain": {"primary_color": "#AA151B", "secondary_color": "#F1BF00", "code": "ES"},
    "England": {"primary_color": "#FFFFFF", "secondary_color": "#C8102E", "code": "GB"},
    "Germany": {"primary_color": "#000000", "secondary_color": "#DD0000", "code": "DE"},
    "Portugal": {"primary_color": "#006600", "secondary_color": "#FF0000", "code": "PT"},
    "Netherlands": {"primary_color": "#FF6600", "secondary_color": "#FFFFFF", "code": "NL"},
    "Croatia": {"primary_color": "#D00000", "secondary_color": "#FFFFFF", "code": "HR"},
    "Belgium": {"primary_color": "#000000", "secondary_color": "#FFD90C", "code": "BE"},
    "Uruguay": {"primary_color": "#6CABDD", "secondary_color": "#FFFFFF", "code": "UY"},
    "Colombia": {"primary_color": "#FCD116", "secondary_color": "#003893", "code": "CO"},
    "Mexico": {"primary_color": "#006847", "secondary_color": "#CE1126", "code": "MX"},
    "United States": {"primary_color": "#3C3B6E", "secondary_color": "#B22234", "code": "US"},
    "Canada": {"primary_color": "#FF0000", "secondary_color": "#FFFFFF", "code": "CA"},
    "Morocco": {"primary_color": "#C1272D", "secondary_color": "#006233", "code": "MA"},
    "Senegal": {"primary_color": "#00853F", "secondary_color": "#FDEF42", "code": "SN"},
    "Japan": {"primary_color": "#FFFFFF", "secondary_color": "#BC002D", "code": "JP"},
    "South Korea": {"primary_color": "#FFFFFF", "secondary_color": "#C60C30", "code": "KR"},
    "Australia": {"primary_color": "#012169", "secondary_color": "#FFCD00", "code": "AU"},
    "Iran": {"primary_color": "#239F40", "secondary_color": "#DA0000", "code": "IR"},
    "Saudi Arabia": {"primary_color": "#006C35", "secondary_color": "#FFFFFF", "code": "SA"},
    "Qatar": {"primary_color": "#8A1538", "secondary_color": "#FFFFFF", "code": "QA"},
    "Ghana": {"primary_color": "#FCD116", "secondary_color": "#006B3F", "code": "GH"},
    "Cameroon": {"primary_color": "#007A5E", "secondary_color": "#CE1126", "code": "CM"},
    "Tunisia": {"primary_color": "#E70013", "secondary_color": "#FFFFFF", "code": "TN"},
    "Egypt": {"primary_color": "#CE1126", "secondary_color": "#000000", "code": "EG"},
    "Nigeria": {"primary_color": "#008751", "secondary_color": "#FFFFFF", "code": "NG"},
    "Switzerland": {"primary_color": "#FF0000", "secondary_color": "#FFFFFF", "code": "CH"},
    "Denmark": {"primary_color": "#C60C30", "secondary_color": "#FFFFFF", "code": "DK"},
    "Poland": {"primary_color": "#FFFFFF", "secondary_color": "#DC143C", "code": "PL"},
    "Austria": {"primary_color": "#ED2939", "secondary_color": "#FFFFFF", "code": "AT"},
    "Serbia": {"primary_color": "#C6363C", "secondary_color": "#0C4076", "code": "RS"},
    "Ecuador": {"primary_color": "#FFDD00", "secondary_color": "#034EA2", "code": "EC"},
    "Paraguay": {"primary_color": "#D52B1E", "secondary_color": "#0038A8", "code": "PY"},

    "Albania": {"primary_color": "#D80027", "secondary_color": "#000000", "code": "AL"},
    "Algeria": {"primary_color": "#006233", "secondary_color": "#FFFFFF", "code": "DZ"},
    "Angola": {"primary_color": "#D21034", "secondary_color": "#000000", "code": "AO"},
    "Bolivia": {"primary_color": "#007A33", "secondary_color": "#FFD700", "code": "BO"},
    "Burkina Faso": {"primary_color": "#EF2B2D", "secondary_color": "#009E49", "code": "BF"},
    "Cape Verde Islands": {"primary_color": "#003893", "secondary_color": "#FFFFFF", "code": "CV"},
    "Chile": {"primary_color": "#D52B1E", "secondary_color": "#0039A6", "code": "CL"},
    "Congo DR": {"primary_color": "#00A3E0", "secondary_color": "#EF3340", "code": "CD"},
    "Costa Rica": {"primary_color": "#CE1126", "secondary_color": "#002B7F", "code": "CR"},
    "Czech Republic": {"primary_color": "#D7141A", "secondary_color": "#11457E", "code": "CZ"},
    "Côte d'Ivoire": {"primary_color": "#F77F00", "secondary_color": "#009E60", "code": "CI"},
    "Equatorial Guinea": {"primary_color": "#E00000", "secondary_color": "#FFFFFF", "code": "GQ"},
    "Finland": {"primary_color": "#003580", "secondary_color": "#FFFFFF", "code": "FI"},
    "Gambia": {"primary_color": "#CE1126", "secondary_color": "#3A7728", "code": "GM"},
    "Georgia": {"primary_color": "#FFFFFF", "secondary_color": "#E41E20", "code": "GE"},
    "Guinea": {"primary_color": "#CE1126", "secondary_color": "#FCD116", "code": "GN"},
    "Guinea-Bissau": {"primary_color": "#FCD116", "secondary_color": "#CE1126", "code": "GW"},
    "Hungary": {"primary_color": "#436F4D", "secondary_color": "#CD2A3E", "code": "HU"},
    "Iceland": {"primary_color": "#003897", "secondary_color": "#D72828", "code": "IS"},
    "Italy": {"primary_color": "#0066CC", "secondary_color": "#FFFFFF", "code": "IT"},
    "Jamaica": {"primary_color": "#009B3A", "secondary_color": "#FED100", "code": "JM"},
    "Mali": {"primary_color": "#14B53A", "secondary_color": "#FCD116", "code": "ML"},
    "Mauritania": {"primary_color": "#006233", "secondary_color": "#D4AF37", "code": "MR"},
    "Mozambique": {"primary_color": "#007168", "secondary_color": "#000000", "code": "MZ"},
    "Namibia": {"primary_color": "#003580", "secondary_color": "#D21034", "code": "NA"},
    "North Macedonia": {"primary_color": "#D20000", "secondary_color": "#F8D80E", "code": "MK"},
    "Panama": {"primary_color": "#005293", "secondary_color": "#D21034", "code": "PA"},
    "Peru": {"primary_color": "#D91023", "secondary_color": "#FFFFFF", "code": "PE"},
    "Romania": {"primary_color": "#002B7F", "secondary_color": "#FCD116", "code": "RO"},
    "Russia": {"primary_color": "#FFFFFF", "secondary_color": "#0039A6", "code": "RU"},
    "Scotland": {"primary_color": "#0065BD", "secondary_color": "#FFFFFF", "code": "GB-SCT"},
    "Slovakia": {"primary_color": "#0B4EA2", "secondary_color": "#EE1C25", "code": "SK"},
    "Slovenia": {"primary_color": "#0056A3", "secondary_color": "#FFFFFF", "code": "SI"},
    "South Africa": {"primary_color": "#007749", "secondary_color": "#FFB81C", "code": "ZA"},
    "Sweden": {"primary_color": "#005293", "secondary_color": "#FECB00", "code": "SE"},
    "Tanzania": {"primary_color": "#1EB53A", "secondary_color": "#00A3DD", "code": "TZ"},
    "Turkey": {"primary_color": "#E30A17", "secondary_color": "#FFFFFF", "code": "TR"},
    "Ukraine": {"primary_color": "#0057B7", "secondary_color": "#FFD700", "code": "UA"},
    "Venezuela": {"primary_color": "#8B1538", "secondary_color": "#F4C300", "code": "VE"},
    "Wales": {"primary_color": "#D30731", "secondary_color": "#046A38", "code": "GB-WLS"},
    "Zambia": {"primary_color": "#198A00", "secondary_color": "#FF7900", "code": "ZM"}
}

In [6]:
# ============================================================
# 6. CREATE ASSETS TABLE
# ============================================================

assets_rows = []

for team in teams:

    info = team_info.get(
        team,
        {
            "primary_color": "#222222",
            "secondary_color": "#F5F5F5",
            "code": None
        }
    )

    file_name = clean_name(team)

    assets_rows.append({
        "team": team,
        "primary_color": info["primary_color"],
        "secondary_color": info["secondary_color"],
        "country_code": info["code"],
        "flag_path": f"assets/flags/{file_name}.png"
    })

assets = pd.DataFrame(assets_rows)

assets.head()

,team,primary_color,secondary_color,country_code,flag_path
0,Albania,#D80027,#000000,AL,assets/flags/albania.png
1,Algeria,#006233,#FFFFFF,DZ,assets/flags/algeria.png
2,Angola,#D21034,#000000,AO,assets/flags/angola.png
3,Argentina,#75AADB,#FFFFFF,AR,assets/flags/argentina.png
4,Australia,#012169,#FFCD00,AU,assets/flags/australia.png


In [7]:
# ============================================================
# 7. DOWNLOAD FLAGS
# ============================================================

for _, row in assets.iterrows():

    team = row["team"]
    code = row["country_code"]

    if pd.isna(code):
        print(f"✗ Missing country code: {team}")
        continue

    url = f"https://flagcdn.com/w320/{str(code).lower()}.png"

    try:
        response = requests.get(url)

        if response.status_code == 200:

            file_name = clean_name(team)

            with open(flags_path / f"{file_name}.png", "wb") as f:
                f.write(response.content)

            print(f"✓ Downloaded flag: {team}")

        else:
            print(f"✗ Failed flag: {team}")

    except Exception as e:
        print(f"ERROR {team}: {e}")

✓ Downloaded flag: Albania
✓ Downloaded flag: Algeria
✓ Downloaded flag: Angola
✓ Downloaded flag: Argentina
✓ Downloaded flag: Australia
✓ Downloaded flag: Austria
✓ Downloaded flag: Belgium
✓ Downloaded flag: Bolivia
✓ Downloaded flag: Brazil
✓ Downloaded flag: Burkina Faso
✓ Downloaded flag: Cameroon
✓ Downloaded flag: Canada
✓ Downloaded flag: Cape Verde Islands
✓ Downloaded flag: Chile
✓ Downloaded flag: Colombia
✓ Downloaded flag: Congo DR
✓ Downloaded flag: Costa Rica
✓ Downloaded flag: Croatia
✓ Downloaded flag: Czech Republic
✓ Downloaded flag: Côte d'Ivoire
✓ Downloaded flag: Denmark
✓ Downloaded flag: Ecuador
✓ Downloaded flag: Egypt
✓ Downloaded flag: England
✓ Downloaded flag: Equatorial Guinea
✓ Downloaded flag: France
✓ Downloaded flag: Gambia
✓ Downloaded flag: Georgia
✓ Downloaded flag: Germany
✓ Downloaded flag: Ghana
✓ Downloaded flag: Guinea
✓ Downloaded flag: Guinea-Bissau
✓ Downloaded flag: Hungary
✓ Downloaded flag: Iran
✓ Downloaded flag: Italy
✓ Downloaded flag

In [8]:
# ============================================================
# 8. CHECK FLAGS EXIST
# ============================================================

assets["flag_exists"] = assets["flag_path"].apply(
    lambda x: os.path.exists("streamlit_app/" + x)
)

assets[
    assets["flag_exists"] == False
]

,team,primary_color,secondary_color,country_code,flag_path,flag_exists


In [9]:
# ============================================================
# 9. SAVE ASSETS
# ============================================================

assets.to_csv(
    "assets.csv",
    index=False
)

print("assets.csv saved successfully!")
print(assets.shape)

assets.head()

assets.csv saved successfully!
(71, 6)


,team,primary_color,secondary_color,country_code,flag_path,flag_exists
0,Albania,#D80027,#000000,AL,assets/flags/albania.png,True
1,Algeria,#006233,#FFFFFF,DZ,assets/flags/algeria.png,True
2,Angola,#D21034,#000000,AO,assets/flags/angola.png,True
3,Argentina,#75AADB,#FFFFFF,AR,assets/flags/argentina.png,True
4,Australia,#012169,#FFCD00,AU,assets/flags/australia.png,True


In [10]:
# ============================================================
# CHECK TEAMS WITHOUT COUNTRY CODE
# ============================================================

assets[
    assets["country_code"].isna()
][["team", "country_code"]]

,team,country_code


In [11]:
# ============================================================
# CHECK MISSING FLAGS
# ============================================================

assets["flag_exists"] = assets["flag_path"].apply(
    lambda x: os.path.exists("streamlit_app/" + x)
)

assets[
    assets["flag_exists"] == False
][["team", "country_code", "flag_path"]]

,team,country_code,flag_path
